# Nail segmentation → TensorFlow Lite (Colab outline)

**Goal:** Train a binary nail mask model and export **`nail_seg.tflite`** for Android (`NailSegmentationHelper`).

**Data layout:**
```
data/
  images/   img_001.jpg, ...
  masks/    img_001.png, ...   # grayscale or RGB; nail = white/high, background = black
```

**Flow:** paths → `tf.data` → **MobileNetV3-Small U-Net** (skip connections at strides **32 / 16 / 8 / optionally 4**) → Dice+BCE → val IoU → **SavedModel** → **TFLite float32** → verify interpreter shapes.

### Open-source dataset (Kaggle)

This notebook can load **[Nail Segmentation Dataset](https://www.kaggle.com/datasets/muhammadhammad261/nail-segmentation-dataset)** (`muhammadhammad261/nail-segmentation-dataset`). After download, folder names may differ (`images`/`masks`, `Images`/`Masks`, `train`/`train`, etc.) — use **Section 2b** and automatic **discovery** in Section 3.

> If the zip layout differs from what we guessed, run `!find /content/nail-data -type d | head -40` once and set `IMAGE_DIR` / `MASK_DIR` manually.

## 1. Dependencies

Colab includes TensorFlow; uncomment to pin a version.

In [ ]:
# !pip install -q "tensorflow>=2.13,<2.17"
# !pip install -q kaggle   # only if using Kaggle download (Section 2b)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from typing import Tuple

print("TensorFlow", tf.__version__)

## 2. Config

**`IMG_SIZE`** must match TFLite input (e.g. 256). Android resizes photos to the model input shape.

- **`DATA_ROOT`**: folder that contains (or will contain) image/mask subfolders after unzip.
- Use **Section 2b** to pull the Kaggle nail dataset into `/content/nail-data`, or upload your own zip and set **`DATA_ROOT`** accordingly.

In [ ]:
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 60
VAL_SPLIT = 0.15
SEED = 42

tf.keras.utils.set_random_seed(SEED)

# Default: where Section 2b places the Kaggle dataset (change if you unzip elsewhere)
DATA_ROOT = Path("/content/nail-data")

# If discovery fails, set these manually after inspecting folders, e.g.:
# IMAGE_DIR = DATA_ROOT / "train" / "images"
# MASK_DIR = DATA_ROOT / "train" / "masks"
IMAGE_DIR = None  # filled by discover_pair_dirs() in Section 3
MASK_DIR = None

## 2b. (Optional) Download from Kaggle

Dataset: **[nail-segmentation-dataset](https://www.kaggle.com/datasets/muhammadhammad261/nail-segmentation-dataset)** (`muhammadhammad261/nail-segmentation-dataset`).

1. On Kaggle: **Account → API → Create New API Token** → download `kaggle.json`.
2. In Colab, upload the file, then run the cell below (it extracts to **`/content/nail-data`** and sets **`DATA_ROOT`** if you re-run the config cell, or you can set `DATA_ROOT = Path("/content/nail-data")` only).

If you already unzipped the dataset manually, **skip** this cell and only set **`DATA_ROOT`** to that folder.

In [ ]:
# --- Optional: Kaggle download — run once per session ---
# !pip install -q kaggle
import os
import shutil
import subprocess
from pathlib import Path

KAGGLE_DATASET = "muhammadhammad261/nail-segmentation-dataset"
NAIL_DATA = Path("/content/nail-data")

try:
    from google.colab import files
except ImportError:
    files = None

if not (Path("/root/.kaggle/kaggle.json").exists()):
    if files is None:
        raise RuntimeError("Place /root/.kaggle/kaggle.json or run in Google Colab")
    print("Upload kaggle.json from Kaggle settings → API")
    up = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)

NAIL_DATA.mkdir(parents=True, exist_ok=True)
subprocess.run(
    ["kaggle", "datasets", "download", "-d", KAGGLE_DATASET, "-p", str(NAIL_DATA), "--unzip"],
    check=True,
)

# Nested zip(s): unzip any remaining .zip inside
import zipfile
for z in list(NAIL_DATA.rglob("*.zip")):
    try:
        with zipfile.ZipFile(z, "r") as zf:
            zf.extractall(z.parent)
        z.unlink(missing_ok=True)
    except zipfile.BadZipFile:
        pass

print("Top-level:", [p.name for p in sorted(NAIL_DATA.iterdir())][:20])
DATA_ROOT = NAIL_DATA
print("Set DATA_ROOT =", DATA_ROOT, "— re-run Section 2 config cell if needed")

In [ ]:
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".webp"}

IMAGE_DIR_HINTS = (
    "images",
    "Images",
    "image",
    "JPEGImages",
    "imgs",
    "train",
    "Training",
)
MASK_DIR_HINTS = (
    "masks",
    "Masks",
    "mask",
    "SegmentationClass",
    "labels",
    "Labels",
    "train",
    "Training",
)


def count_pairs_between(img_dir: Path, mask_dir: Path) -> int:
    return len(list_pairs(img_dir, mask_dir))


def discover_pair_dirs(root: Path) -> Tuple[Path, Path]:
    """Find image/ mask folders under root with matching filenames (by stem)."""
    subs = [root]
    try:
        subs.extend([p for p in root.rglob("*") if p.is_dir()])
    except OSError:
        pass
    best = (None, None, -1)
    for parent in subs:
        dirs = [p for p in parent.iterdir() if p.is_dir()]
        for img_candidate in dirs:
            name_low = img_candidate.name.lower()
            if not any(
                h.lower() in name_low or name_low == h.lower()
                for h in IMAGE_DIR_HINTS
            ):
                continue
            for mask_candidate in dirs:
                if mask_candidate == img_candidate:
                    continue
                mn = mask_candidate.name.lower()
                if not any(
                    h.lower() in mn or mn == h.lower() for h in MASK_DIR_HINTS
                ):
                    continue
                n = count_pairs_between(img_candidate, mask_candidate)
                if n > best[2]:
                    best = (img_candidate, mask_candidate, n)
    if best[2] <= 0:
        raise RuntimeError(
            "Could not find matching images/masks under DATA_ROOT. "
            "Run: !find DATA_ROOT -type d | head -50 "
            "and set IMAGE_DIR / MASK_DIR manually in Section 2."
        )
    print("Discovered IMAGE_DIR:", best[0])
    print("Discovered MASK_DIR:", best[1])
    print("Matching pairs:", best[2])
    return best[0], best[1]


def list_pairs(image_dir: Path, mask_dir: Path):
    pairs = []
    for p in sorted(image_dir.iterdir()):
        if p.suffix.lower() not in IMAGE_EXT:
            continue
        stem = p.stem
        m = None
        for ext in (".png", ".jpg", ".jpeg"):
            cand = mask_dir / f"{stem}{ext}"
            if cand.exists():
                m = cand
                break
        if m is None:
            cand_guess = mask_dir / (stem + "_mask.png")
            if cand_guess.exists():
                m = cand_guess
        if m is None:
            continue
        pairs.append((str(p), str(m)))
    return pairs


if IMAGE_DIR is None or MASK_DIR is None:
    IMAGE_DIR, MASK_DIR = discover_pair_dirs(DATA_ROOT)

pairs = list_pairs(IMAGE_DIR, MASK_DIR)
print("pairs:", len(pairs))
assert len(pairs) > 0, "No pairs — fix DATA_ROOT or set IMAGE_DIR / MASK_DIR manually"

## 4. `tf.data` pipeline (pure TF decode)

Masks: decode PNG → grayscale → **resize nearest** → `[0,1]`.  
RGB images: **resize bilinear** → `[0,1]`.

In [ ]:
def load_sample(img_path, mask_path):
    img_data = tf.io.read_file(img_path)
    img = tf.image.decode_image(img_data, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)

    mask_raw = tf.io.read_file(mask_path)
    mask_rgb = tf.image.decode_image(mask_raw, channels=3, expand_animations=False)
    mask_rgb = tf.image.convert_image_dtype(mask_rgb, tf.float32)
    # Grayscale / binary mask (handles RGB mask PNGs too)
    mask = (
        0.299 * mask_rgb[..., 0:1]
        + 0.587 * mask_rgb[..., 1:2]
        + 0.114 * mask_rgb[..., 2:3]
    )
    mask = tf.clip_by_value(mask, 0.0, 1.0)

    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], method="bilinear")
    mask = tf.image.resize(mask, [IMG_SIZE, IMG_SIZE], method="nearest")
    mask = tf.clip_by_value(mask, 0.0, 1.0)
    return img, mask

def augment(img, mask):
    seed = tf.random.uniform([2], maxval=10_000, dtype=tf.int32)
    img = tf.image.stateless_random_flip_left_right(img, seed=seed)
    mask = tf.image.stateless_random_flip_left_right(mask, seed=seed)
    return img, mask

def make_ds(paths_pairs, training: bool):
    ips = [a for a, _ in paths_pairs]
    mps = [b for _, b in paths_pairs]
    ds = tf.data.Dataset.from_tensor_slices((ips, mps))
    if training:
        ds = ds.shuffle(min(500, len(paths_pairs)), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_sample, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

n = len(pairs)
n_val = max(1, int(n * VAL_SPLIT))
pairs_train = pairs[:-n_val]
pairs_val = pairs[-n_val:]

train_ds = make_ds(pairs_train, training=True)
val_ds = make_ds(pairs_val, training=False)

print("train batches:", len(pairs_train), "val:", len(pairs_val))

## 5. Loss + batch IoU

`iou_metric` reduces **mean IoU per image** in the batch.

In [ ]:
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [tf.shape(y_true)[0], -1])
    y_pred_f = tf.reshape(y_pred, [tf.shape(y_pred)[0], -1])
    inter = tf.reduce_sum(y_true_f * y_pred_f, axis=-1)
    denom = tf.reduce_sum(y_true_f, axis=-1) + tf.reduce_sum(y_pred_f, axis=-1)
    dice = (2.0 * inter + smooth) / (denom + smooth)
    return 1.0 - tf.reduce_mean(dice)

def combined_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return tf.reduce_mean(bce) + dice_loss(y_true, y_pred)

def iou_metric(y_true, y_pred):
    pred_bin = tf.cast(y_pred > 0.5, tf.float32)
    inter = tf.reduce_sum(y_true * pred_bin, axis=[1, 2, 3])
    union = tf.reduce_sum(tf.maximum(y_true, pred_bin), axis=[1, 2, 3])
    iou = inter / (union + 1e-6)
    return tf.reduce_mean(iou)

## 6. Model — MobileNetV3-Small **U-Net with skip connections**

- Scan backbone layers and keep the **last tensor at each stride** `{32, 16, 8, 4}` (spatial grid `IMG_SIZE/stride`). Stride **4** may be absent on some TF builds — decoder falls back without that skip.
- **Decoder:** upsample with **`Conv2DTranspose`**, **concatenate** skip, **`Conv2D`** to mix channels. Repeat until **`IMG_SIZE`** is reached (typically **five** ×2 upsamples from stride **32**).
- Skips restore **fine boundaries** (cuticle, free edge) vs a plain decoder.

In [ ]:
def collect_stride_tensors(base: keras.Model, img_size: int):
    """Last layer output per stride (relative to full-res input)."""
    best = {}
    for layer in base.layers:
        os = layer.output_shape
        if not isinstance(os, tuple) or len(os) != 4:
            continue
        _, h, w, _ = os
        if h is None or w is None:
            continue
        hi, wi = int(h), int(w)
        if hi != wi or hi == 0:
            continue
        if img_size % hi != 0:
            continue
        stride = img_size // hi
        if stride in (4, 8, 16, 32):
            best[stride] = layer.output
    return best


def conv_block(x, filters: int, name=None):
    x = layers.Conv2D(filters, 3, padding="same", activation="relu", name=name)(x)
    return x


def build_model(img_size: int) -> keras.Model:
    assert img_size % 32 == 0, "Use IMG_SIZE divisible by 32 (e.g. 256, 320)"
    inputs = keras.Input(shape=(img_size, img_size, 3))
    base = keras.applications.MobileNetV3Small(
        input_tensor=inputs,
        include_top=False,
        weights="imagenet",
        alpha=1.0,
        minimalistic=False,
        include_preprocessing=False,
    )
    skips = collect_stride_tensors(base, img_size)
    for s in (32, 16, 8):
        assert s in skips, (
            f"Need stride-{s} skip. Got strides {list(skips.keys())}. "
            "Try printing [l.name, l.output_shape] for base.layers."
        )
    if 4 not in skips:
        print("Note: no stride-4 skip found — decoder continues without it.")

    print("Skip strides used:", sorted(skips.keys()))

    x = skips[32]

    x = layers.Conv2DTranspose(256, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.Concatenate()([x, skips[16]])
    x = conv_block(x, 256, "dec16")

    x = layers.Conv2DTranspose(128, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.Concatenate()([x, skips[8]])
    x = conv_block(x, 128, "dec8")

    x = layers.Conv2DTranspose(64, 3, strides=2, padding="same", activation="relu")(x)
    if 4 in skips:
        x = layers.Concatenate()([x, skips[4]])
    x = conv_block(x, 64, "dec4")

    x = layers.Conv2DTranspose(32, 3, strides=2, padding="same", activation="relu")(x)
    x = conv_block(x, 32, "up2")

    x = layers.Conv2DTranspose(16, 3, strides=2, padding="same", activation="relu")(x)
    x = conv_block(x, 16, "up1")

    out = layers.Conv2D(
        1, 1, activation="sigmoid", dtype="float32", name="nail_prob"
    )(x)
    return keras.Model(inputs, out, name="nail_seg_unet")


model = build_model(IMG_SIZE)
model.summary()

dummy = tf.zeros((1, IMG_SIZE, IMG_SIZE, 3))
assert model(dummy).shape == (1, IMG_SIZE, IMG_SIZE, 1)

### Optional: freeze backbone for early epochs

After `model.summary()`, find the MobileNet layer name (often contains `mobilenet`), then:

```python
enc = model.get_layer("mobilenetv3small")  # adjust to printed name
enc.trainable = False
model.compile(optimizer=keras.optimizers.Adam(1e-4), loss=combined_loss, metrics=[iou_metric])
model.fit(train_ds, validation_data=val_ds, epochs=15)
enc.trainable = True
model.compile(optimizer=keras.optimizers.Adam(1e-5), loss=combined_loss, metrics=[iou_metric])
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)
```

## 7. Train

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=combined_loss,
    metrics=[iou_metric],
)

callbacks = [
    keras.callbacks.EarlyStopping(
        patience=10, restore_best_weights=True, monitor="val_iou_metric", mode="max"
    ),
    keras.callbacks.ModelCheckpoint(
        "best_nail.keras", save_best_only=True, monitor="val_iou_metric", mode="max"
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

## 8. Visual check

In [ ]:
def show_val_sample(idx=0):
    for imgs, masks in val_ds.take(1):
        pred = model.predict(imgs[:1], verbose=0)
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 3, 1)
        plt.imshow(imgs[idx].numpy())
        plt.title("Image")
        plt.axis("off")
        plt.subplot(1, 3, 2)
        plt.imshow(masks[idx].numpy().squeeze(), vmin=0, vmax=1, cmap="gray")
        plt.title("GT mask")
        plt.axis("off")
        plt.subplot(1, 3, 3)
        plt.imshow(pred[idx].squeeze(), vmin=0, vmax=1, cmap="gray")
        plt.title("Predicted")
        plt.axis("off")
        plt.tight_layout()
        plt.show()
        break

# show_val_sample()

## 9. Export SavedModel + TFLite (float32)

Android expects **RGB float32 [0,1]**, **NHWC**, batch **1**. Output: nail probability **[0,1]** per pixel.

In [ ]:
tf.saved_model.save(model, "saved_model_nail")

In [ ]:
converter = tf.lite.TFLiteConverter.from_saved_model("saved_model_nail")
converter.optimizations = []
tflite_bytes = converter.convert()
open("nail_seg.tflite", "wb").write(tflite_bytes)
print("Wrote nail_seg.tflite", len(tflite_bytes), "bytes")

**Alternative:** convert directly from Keras without SavedModel:
```python
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_bytes = converter.convert()
```

## 10. Verify interpreter (same checks as Android)

In [ ]:
interp = tf.lite.Interpreter(model_content=tflite_bytes)
interp.allocate_tensors()
inn = interp.get_input_details()[0]
out = interp.get_output_details()[0]
print("INPUT", inn["shape"], inn["dtype"])
print("OUTPUT", out["shape"], out["dtype"])

test_in = np.zeros((1, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
interp.set_tensor(inn["index"], test_in)
interp.invoke()
print("out sample", interp.get_tensor(out["index"]).shape)

## 11. Ship to Android

1. Download **`nail_seg.tflite`** from Colab.
2. Place at **`app/src/main/assets/nail_seg.tflite`**.
3. Align **`NailSegmentationHelper`**: input `[1, H, W, 3]` float32, `/255` if you trained on `[0,1]` — **match preprocessing**.
4. Parse output tensor shape from **`getOutputTensor(0)`** (single channel vs multi-class).

**Debugging skips:** if `assert s in skips` fails, run:
```python
for ly in model.get_layer("mobilenet_v3_small").layers:
    print(ly.name, ly.output_shape)
```
and adjust **`collect_stride_tensors`** to pick named layers (e.g. `block_*_expand`).